# Stage 3B · YOLO26 screening

RoadGuard AI · Matched validation review

## 1. Frozen snapshot and screening protocol

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

from roadguard.utils.paths import project_root

root = project_root()
experiments = root / "artifacts/experiments"
comparison_path = experiments / "stage3b_comparison.json"
report = (
    json.loads(comparison_path.read_text(encoding="utf-8")) if comparison_path.is_file() else None
)
if report:
    print("Snapshot", report["dataset_fingerprint"])
    display(
        pd.Series(
            {
                key: report["training_protocol"][key]
                for key in (
                    "epochs",
                    "image_size",
                    "batch_size",
                    "seed",
                    "workers",
                    "amp",
                    "optimizer",
                )
            },
            name="Matched training",
        )
    )
else:
    print("The comparison appears after both YOLO26 runs finish and pass the integrity audit.")

## 2. Validation accuracy and per-class AP

In [ ]:
if report:
    models = pd.DataFrame(report["models"]).set_index("model")
    display(
        models[
            [
                "map50_95",
                "map50",
                "precision",
                "recall",
                "D40_recall",
                "best_epoch",
            ]
        ].round(4)
    )
    display(
        models[
            [
                "D00_ap50_95",
                "D10_ap50_95",
                "D20_ap50_95",
                "D40_ap50_95",
            ]
        ].round(4)
    )

## 3. Detector speed and resource use

Latency is a batched validation average on the RTX 4070 Laptop GPU. Estimated FPS describes detector throughput. Training VRAM reports the highest sampled CUDA reservation in the training log.

In [ ]:
if report:
    display(
        models[
            [
                "parameter_count",
                "checkpoint_bytes",
                "peak_logged_training_vram_gib",
                "preprocess_latency_ms",
                "inference_latency_ms",
                "postprocess_latency_ms",
                "total_latency_ms",
                "estimated_detector_fps",
            ]
        ].round(3)
    )
    axis = models.plot.scatter(x="total_latency_ms", y="map50_95", figsize=(8, 4))
    for name, row in models.iterrows():
        axis.annotate(
            name,
            (row["total_latency_ms"], row["map50_95"]),
            xytext=(5, 5),
            textcoords="offset points",
        )
    axis.set(xlabel="Total validation latency (ms/image)", ylabel="Validation mAP50–95")

## 4. Checkpoint evidence

Select a model to view the training curves, confusion matrix and precision–recall curve saved with its checkpoint.

In [ ]:
if report:
    selected_model = report["recommendation"]["model"]
    # Set selected_model to another model name to inspect its plots.
    directory = experiments / models.loc[selected_model, "run"]
    for name in ("results.png", "confusion_matrix.png", "BoxPR_curve.png"):
        display(Image(filename=str(directory / name)))

## 5. Architecture recommendation

In [ ]:
if report:
    recommendation = report["recommendation"]
    print("Recommended architecture", recommendation["model"])
    print(recommendation["rationale"])
    print("Longer training justified", recommendation["longer_training_justified"])
    print("Test evaluation performed", report["test_evaluated"])